# RF-DETR Object Detection with OpenVINO

RF-DETR is a real-time transformer-based object detector built on a DINOv2 vision backbone and a deformable DETR decoder. This tutorial shows how to export RF-DETR from the [Hugging Face model collection](https://huggingface.co/collections/Roboflow/rf-detr) and run object detection with OpenVINO.

The default Nano model keeps the download and conversion practical. Small, Medium, Base, and Large variants can be selected for higher accuracy.

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and is using a custom branch of optimum-intel. It may be fully supported and validated in the future.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select a model](#Select-a-model)
- [Convert the model](#Convert-the-model)
- [Run object detection](#Run-object-detection)
- [Interactive demo](#Interactive-demo)

### References

- [RF-DETR paper](https://arxiv.org/abs/2511.09554)
- [RF-DETR repository](https://github.com/roboflow/rf-detr)
- [OpenVINO documentation](https://docs.openvino.ai/)

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/rf-detr-object-detection/rf-detr-object-detection.ipynb" />
### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top](#Table-of-contents)

Install the runtime dependencies and the Optimum Intel RF-DETR feature branch. Transformers is installed after Optimum Intel because RF-DETR requires Transformers 5.10, while the current package metadata still declares an older upper bound.

In [ ]:
from pathlib import Path

import requests

for helper_name in ("notebook_utils.py", "pip_helper.py"):
    helper_path = Path(helper_name)
    if not helper_path.exists():
        response = requests.get(
            f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{helper_name}",
            timeout=30,
        )
        response.raise_for_status()
        helper_path.write_text(response.text, encoding="utf-8")

from pip_helper import pip_install

pip_install(
    "-q",
    "torch>=2.10",
    "torchvision",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)
pip_install(
    "-q",
    "optimum>=2.3.0,<3.0",
    "openvino>=2026.0",
    "Pillow",
    "ipywidgets",
    "requests>=2.33,<3.0",
    "gradio>=4.19,<6",
)
pip_install(
    "-q",
    "--no-deps",
    "git+https://github.com/aleksandr-mokrov/optimum-intel.git@fix/rf-detr-followup-1843",
)
pip_install("-q", "transformers>=5.10,<5.11")

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("rf-detr-object-detection.ipynb")

In [ ]:
import inspect

import optimum.intel
import transformers

from notebook_utils import device_widget, download_file

optimum_source = Path(inspect.getfile(optimum.intel)).resolve()
print(f"Optimum Intel module: {optimum_source}")
print(f"Transformers version: {transformers.__version__}")

IMAGE_PATH = Path("data/coco_bike.jpg")
if not IMAGE_PATH.exists():
    download_file(
        url="https://storage.openvinotoolkit.org/repositories/openvino_notebooks/data/data/image/coco_bike.jpg",
        filename=IMAGE_PATH.name,
        directory=IMAGE_PATH.parent,
    )

## Select a model
[back to top](#Table-of-contents)

The RF-DETR checkpoints below use the Apache-2.0 license. Nano is selected by default for a faster first run. Larger variants require more download time, memory, and conversion time.

| Variant | Export resolution |
|---|---:|
| Nano | 384 × 384 |
| Small | 512 × 512 |
| Medium | 576 × 576 |
| Base | 560 × 560 |
| Large | 704 × 704 |

The exporter derives the valid resolution from each model configuration. RF-DETR spatial dimensions must be divisible by `patch_size × num_windows`.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

MODEL_OPTIONS = [
    ("RF-DETR Nano (384 × 384)", "Roboflow/rf-detr-nano"),
    ("RF-DETR Small (512 × 512)", "Roboflow/rf-detr-small"),
    ("RF-DETR Medium (576 × 576)", "Roboflow/rf-detr-medium"),
    ("RF-DETR Base (560 × 560)", "Roboflow/rf-detr-base"),
    ("RF-DETR Large (704 × 704)", "Roboflow/rf-detr-large"),
]

model_selector = widgets.Dropdown(options=MODEL_OPTIONS, value=MODEL_OPTIONS[0][1], description="Model:")
device = device_widget(default="CPU", exclude=["NPU"])
display(model_selector, device)

## Convert the model
[back to top](#Table-of-contents)

`OVModelForObjectDetection.from_pretrained(..., export=True)` loads the selected Transformers checkpoint, traces it with model-specific dummy inputs, and converts it to OpenVINO IR. The exported model is cached in the local `model` directory and reused on subsequent runs.

In [ ]:
import gc

from optimum.intel import OVModelForObjectDetection
from transformers import AutoImageProcessor

MODEL_ID = model_selector.value
MODEL_DIR = Path("model") / f"{MODEL_ID.split('/')[-1]}-ov"
MODEL_PATH = MODEL_DIR / "openvino_model.xml"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)

if not MODEL_PATH.exists():
    print(f"Converting {MODEL_ID} to OpenVINO IR...")
    exported_model = OVModelForObjectDetection.from_pretrained(MODEL_ID, export=True, compile=False)
    exported_model.save_pretrained(MODEL_DIR)
    del exported_model
    gc.collect()
else:
    print(f"Using cached OpenVINO model from {MODEL_DIR}")

ov_model = OVModelForObjectDetection.from_pretrained(MODEL_DIR, device=device.value)
print(f"Loaded {MODEL_ID} on {device.value}")

## Run object detection
[back to top](#Table-of-contents)

The image processor resizes and normalizes the image and creates the pixel mask required by RF-DETR. After OpenVINO inference, the same processor converts normalized center-format boxes into pixel coordinates.

In [ ]:
confidence_threshold = widgets.FloatSlider(
    value=0.4,
    min=0.05,
    max=0.95,
    step=0.05,
    description="Threshold:",
    readout_format=".2f",
)
confidence_threshold

In [ ]:
from PIL import Image

from gradio_helper import run_object_detection

image = Image.open(IMAGE_PATH)
visualization, detections = run_object_detection(
    ov_model,
    processor,
    image,
    confidence_threshold.value,
)

print(f"Detected {len(detections)} objects with confidence >= {confidence_threshold.value:.2f}")
visualization

## Interactive demo
[back to top](#Table-of-contents)

Upload an image and adjust the confidence threshold to run RF-DETR interactively with the selected OpenVINO device.

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_model, processor, IMAGE_PATH)

try:
    demo.launch(debug=False)
except Exception:
    demo.launch(debug=False, share=True)